In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch.nn as nn

import os
import zipfile
import urllib.request

from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

from scipy.sparse import csr_matrix

from sklearn.preprocessing import LabelEncoder

In [ ]:
data_dir = "../data"

## Data downloading

In [ ]:
movies_data_dir_path = os.path.join(data_dir, "ml-1m")
zip_path = os.path.join(data_dir, "ml-1m.zip")
data_url = "http://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    
if not os.path.exists(movies_data_dir_path):
    print("there is no data. downloading")
    
    urllib.request.urlretrieve(data_url, zip_path)
    
    with zipfile.ZipFile(zip_path, "r") as zip_data:
        zip_data.extractall(data_dir)
        
        os.remove(zip_path)
        print(f'data is downloaded and extracted to {movies_data_dir_path}')
else:
    print(f'data already exists in {movies_data_dir_path}')

In [ ]:
ratings_path  = os.path.join(movies_data_dir_path, 'ratings.dat')
col_names = ['user_id', 'item_id', 'rating', 'timestamp']

df = pd.read_csv(ratings_path, sep='::', engine='python', names=col_names)
df.head()

## EDA

In [ ]:
print(f'interactions quantity: {df.shape}')
print(f'unique users: {len(df['user_id'].unique())}')
print(f'unique items: {len(df['item_id'].unique())}')

In [ ]:
def rating_quantity(main_data, group_by_col, func_col, func):
    group_df = main_data.groupby(group_by_col).agg(
        quantity=(func_col, func)
    ).reset_index()
    
    group_df = group_df.rename(columns={
        'quantity':f'{func_col}_quantity'
    })
    
    group_df = group_df.sort_values(by=f'{func_col}_quantity', ascending=False)
    
    return group_df

user_rating = rating_quantity(df, 'user_id', 'rating', 'size')
item_rating = rating_quantity(df, 'item_id', 'rating', 'size')

In [ ]:
item_rating.head(10)

In [ ]:
user_rating.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=user_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у пользователей')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=item_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у фильмов')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
min_date = df['timestamp'].min()

df['last_watch_dt'] = df['timestamp'] - min_date
df['last_watch_dt'] = df['last_watch_dt'].apply(lambda x: int(str(x).split()[0]))
df = df.drop(columns='timestamp')

In [ ]:
df['last_watch_dt'].hist(bins=50, grid=False)
plt.show()

## Filtering data

In [ ]:
active_users = user_rating.loc[user_rating['rating_quantity'] >= 10, 'user_id'].unique()
active_films = item_rating.loc[item_rating['rating_quantity'] >= 10, 'item_id'].unique()

mask = (df['user_id'].isin(active_users)) & (df['item_id'].isin(active_films))
filtered_df = df[mask].copy()

In [ ]:
print(f'old shape: {df.shape}')
print(f'filtered df shape: {filtered_df.shape}')

## Train test split

In [ ]:
max_date = filtered_df['last_watch_dt'].max()
log_size = 30

cut_off_date = max_date - log_size 

In [ ]:
train_df = filtered_df[filtered_df['last_watch_dt'] < cut_off_date].copy()
test_df = filtered_df[filtered_df['last_watch_dt'] >= cut_off_date].copy()

In [ ]:
assert train_df['last_watch_dt'].max() < test_df['last_watch_dt'].min(), 'wrong cut off date'

In [ ]:
train_users_ids = train_df['user_id'].unique()
train_item_ids = train_df['item_id'].unique()

test_df = test_df[test_df['user_id'].isin(train_users_ids)].copy()
test_df = test_df[test_df['item_id'].isin(train_item_ids)].copy()

In [ ]:
assert set(test_df['user_id'].unique()).issubset(set(train_df['user_id'].unique())), 'wrong user ids in test df'
assert set(test_df['item_id'].unique()).issubset(set(train_df['item_id'].unique())), 'wrong item ids in test df'

## Modeling

In [ ]:
class ImplicitModel:
    def __init__(self, model):
        self.model = model
        self.trained = False
        
    def fit(self, train_df):
        self.item_encoder = LabelEncoder()
        self.user_encoder = LabelEncoder()
        self.item_encoder.fit(train_df['item_id'])
        self.user_encoder.fit(train_df['user_id'])
        
        self.train_ratings = self.create_matrix(train_df, ['item_id', 'user_id'])
        self.model.fit(self.train_ratings)
        self.trained = True
    
    def predict(self, test_df, top_k = 10):
        if not self.trained:
            raise ValueError("Model is not fitted")
        
        users_to_predict = test_df['user_id'].unique()
        encoded_users_to_predict = self.user_encoder.transform(users_to_predict)
        
        recs = self.model.recommend(
            encoded_users_to_predict, self.train_ratings[encoded_users_to_predict], 
            N=top_k, filter_already_liked_items=False
            )[0]
        
        user_recs = [self.item_encoder.inverse_transform(i) for i in recs]
        
        return user_recs
    
    def create_matrix(self, df, ids_list):
        encoded_item_ids = self.item_encoder.transform(df[ids_list[0]])
        encoded_user_ids = self.user_encoder.transform(df[ids_list[1]])
        
        n_items = len(self.item_encoder.classes_)
        n_users = len(self.user_encoder.classes_)
        
        matrix_shape = (n_users, n_items)
        
        sparse_matrix = csr_matrix((np.ones(len(encoded_user_ids)), (encoded_user_ids, encoded_item_ids)), shape=matrix_shape, dtype=np.float32)
        
        return sparse_matrix
        